In [1]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261027 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

In [2]:
import os
import tempfile
import subprocess

# Create a separate Cloud SDK configuration directory
os.environ["CLOUDSDK_CONFIG"] = tempfile.mkdtemp(
        prefix="gcloud-dsub-",
        dir="/tmp",
)

print("CLOUDSDK_CONFIG:", os.environ["CLOUDSDK_CONFIG"])
print("CLOUDSDK_CONFIG =", os.environ.get("CLOUDSDK_CONFIG", "(default)"))
print("HOME =", os.environ.get("HOME"))

subprocess.run(
        ["gcloud", "config", "get-value", "account"],
        check=False,
)

CLOUDSDK_CONFIG: /tmp/gcloud-dsub-9aoi3bw5
CLOUDSDK_CONFIG = /tmp/gcloud-dsub-9aoi3bw5
HOME = /home/dataproc
pet-27662581260322f686e2a@wb-fast-radish-6415.iam.gserviceaccount.com


CompletedProcess(args=['gcloud', 'config', 'get-value', 'account'], returncode=0)

# COMMAND 

In [3]:
%%writefile convert_plink_to_bgen.sh

# Convert PLINK to BGEN 
awk 'BEGIN{OFS="\t"} NR > 1 {print $2, $2}' "${FAM}" > samplelist.txt

# Check filtering files 
head snplist.txt
head samplelist.txt

# Convert PLINK to BGEN
plink2 \
    --bgen "${GENO}/chr${CHROM}.bgen" ref-last \
    --sample "${GENO}/chr${CHROM}.sample" \
    --extract "${BIM}" \
    --keep samplelist.txt \
    --export bgen-1.2 \
    --out "${OUTPUT}/chr${CHROM}"


Overwriting convert_plink_to_bgen.sh


# SUBMIT COMMAND 

In [10]:
import subprocess
import pandas as pd

chromosome = []
sizes = [] 

# Loop for all chromosome 
for chrom in range(1, 23):
    
    # Extract size of genomic plink bed file in GB for each chromosome [Version 8]
    command = [
        "gsutil",
        "-u", 'wb-cheerful-parsnip-892',
        "ls",
        "-lha",
        f"gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/bgen/chr{chrom}.bgen"
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    
    # Extract genomic size from string 
    lines = result.stdout.splitlines()
    line = lines[0].split()
    
    # Create list genomic size om is STILL undefined (critical failure)
    sizes.append(line[0])
    chromosome.append(chrom)
    
# Deposit genomic file size into data frame 
genomic_size = pd.DataFrame({
    "chromosome": chromosome,
    "size": sizes
})

print(genomic_size)

    chromosome   size
0            1  28.17
1            2  17.54
2            3  14.64
3            4  10.32
4            5  12.36
5            6  12.78
6            7  13.28
7            8  10.32
8            9  10.17
9           10  10.48
10          11  14.25
11          12     14
12          13   5.39
13          14    8.8
14          15  11.14
15          16  12.99
16          17  15.08
17          18   5.11
18          19  20.55
19          20   6.51
20          21   3.69
21          22   6.31


In [11]:
# Logging directory
LOGGING=f"{os.environ['WORKSPACE_BUCKET']}/data/logging"
# Output directory 
OUTPUT=f"{os.environ['WORKSPACE_BUCKET']}/data/staar/bgen/"
# Path to genomic data 
MNT_PATH="/mnt/data/input/gs/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/bgen"

MAX_DISK_SIZE = 500 # Set maximum disk; adjust for acaf or exome accordingly
EXTRA_DISK_SIZE = 20 # Set additional disk size for each chromosome 

# Loop to submit dsub command 
for CHROM in [2, 4, 15]:    
    print(f"Submitting dsub command for chromosome {CHROM}")
    
    DISK_SIZE = round(float(genomic_size.loc[genomic_size['chromosome'] == CHROM, 'size'].iloc[0])) + EXTRA_DISK_SIZE
    
    print(f"Requesting disk size: {DISK_SIZE}")

    if DISK_SIZE > MAX_DISK_SIZE: # Set maximum disk size 
        print(f"Disk too large for chromosome {CHROM}")
        break
        
    if DISK_SIZE > 1000: # Absolute maximum for version 8  
        print(f"Disk too large for chromosome {CHROM}")
        break

    # Variant list 
    BIM=f"{os.environ['WORKSPACE_BUCKET']}/data/wgs_qc/wgs_variant_qc/exome/EUR_exome_chr{CHROM}_variant_qc.snplist"
    # Sample list 
    FAM=f"{os.environ['WORKSPACE_BUCKET']}/data/wgs_qc/analysis_samples_wgs_qc/exome/EUR-like_exome_qc.txt"

    # Version 8 BGEN [Change to Version 9]
    BGEN=f'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/bgen/chr{CHROM}.bgen'
    SAMPLE=f'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/bgen/chr{CHROM}.sample'
    BGI=f'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/bgen/chr{CHROM}.bgen.bgi'

    # Submission command 
    command = [
        "aou_dsub",
        "--project", os.getenv('GOOGLE_CLOUD_PROJECT'), 
        "--user-project", os.getenv('GOOGLE_CLOUD_PROJECT'), 
        "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/gfanz/plink2:latest",
        "--boot-disk-size", "10",
        "--disk-size", str(DISK_SIZE),
        "--machine-type", "n1-standard-1",
        "--logging", LOGGING,
        "--input", f"BIM={BIM}",
        "--input", f"FAM={FAM}",
        "--input", f"BGEN={BGEN}",
        "--input", f"SAMPLE={SAMPLE}",
        "--input", f"BGI={BGI}",
        "--output-recursive", f"OUTPUT={OUTPUT}",
        "--env", f"GENO={MNT_PATH}",
        "--env", f"CHROM={CHROM}",
        "--script", "convert_plink_to_bgen.sh"
    ]

    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {' '.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )

    print(dsub_command.stdout)
    print(dsub_command.stderr)

Submitting dsub command for chromosome 2
Requesting disk size: 38
convert-pl--vienose--261028-125910-91

Job properties:
  job-id: convert-pl--vienose--261028-125910-91
  job-name: convert-plink-to-bgen
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/convert-pl--vienose--261028-125910-91-0-0
Launched job-id: convert-pl--vienose--261028-125910-91
To check the status, run:
  dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'convert-pl--vienose--261028-125910-91' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'convert-pl--vienose--261028-125910-91' --users 'vienose'

Submitting dsub command for chromosome 4
Requesting disk size: 30
convert-pl--vienose--261028-125912-42

Job properties:
  job-id: convert-pl--vienose--261028-125912-42
  job-name: convert-plink-to-bgen
  user-id: vienose
Provid

In [3]:
! dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --users 'vienose' --status '*' |     head 

Job Name         Status    Last Update
---------------  --------  --------------
convert-plin...  SUCCESS   10-28 15:01:41
convert-plin...  SUCCESS   10-28 15:17:18
convert-plin...  SUCCESS   10-28 17:14:47
convert-plin...  SUCCESS   10-28 03:57:28
filtered-hig...  SUCCESS   10-26 13:24:36
filtered-hig...  SUCCESS   10-26 13:24:53
filtered-hig...  SUCCESS   10-26 13:24:42
filtered-hig...  SUCCESS   10-26 13:18:11
Exception ignored in: <_io.TextIOWrapper name='<stdout>' mode='w' encoding='utf-8'>
BrokenPipeError: [Errno 32] Broken pipe


In [51]:
!gcloud storage ls {os.environ['WORKSPACE_BUCKET']}/data/logging | grep 'convert-pl--vienose--261028-004817-31'

gs://rw-migration-aou-rw-9f879fc4/data/logging/convert-pl--vienose--261028-004817-31-stderr.log
gs://rw-migration-aou-rw-9f879fc4/data/logging/convert-pl--vienose--261028-004817-31-stdout.log
gs://rw-migration-aou-rw-9f879fc4/data/logging/convert-pl--vienose--261028-004817-31.log


In [6]:
# !gsutil ls -lha gs://rw-migration-aou-rw-9f879fc4/data/staar/bgen/